In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 필요한 패키지 설치
!pip install -q adapter-transformers
!pip install -q --upgrade transformers accelerate
!pip install -q --upgrade transformers
!pip install -q peft==0.6.0
!pip install -q ipywidgets
!pip install -q datasets

# Library

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import os
import json
from typing import List, Dict

from tqdm import tqdm
tqdm.pandas()

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# pytorch
import torch
import transformers
from transformers import  GenerationConfig,  AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# metric
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Utils

In [ ]:
def generate_aspect_prompts(df: pd.DataFrame) -> List[Dict[str, str]]:
    """
    Generate a list of prompts for summarizing each review across predefined aspects.

    Args:
        df: DataFrame containing the reviews.

    Returns:
        A list of dicts, each with keys "instruction", "input", and "output".
    """
    base_instruction = (
        "Read the following review and generate the profile by explicitly including the reasoning process as part of the output."
        "Make sure the reasoning steps are logically connected and clearly lead to the final profile.\n\n"
    )

    return [
        {
            "instruction": base_instruction,
            "input": str(review),
            "output": ""
        }
        for review in df['text']
    ]

In [ ]:
# 모델에 해야 할 적업을 추가적으로 설명함.
def generate_prompt(instruction, input=None):
    if input:
         return f"""

### Instruction:
{instruction}

### Input:
{input}

### Response:
"""
    else:
        return f"""

### Instruction:
{instruction}

### Response:
"""

In [ ]:
# Chunk 데이터 저장
def save_chunk(data, index, save_dir):
    file_path = os.path.join(save_dir, f"output_chunk_{index:03}.json")
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"✅ Saved: {file_path} ({len(data)} samples)")

# Load Data

In [29]:
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_RQ3_diff.pkl')

df = df[20000:]

print(df.shape)
df.head()

(16856, 9)


In [ ]:
df = df.reset_index(drop=True)

In [ ]:
# 데이터셋에서 글자 수 확인
# df['text'].apply(lambda row : len(row)).describe().astype(int)

In [ ]:
# 데이터셋에서 공백기준 단어의 수 확인
# df['text'].apply(lambda row : len(row.split(' '))).describe().astype(int)

## Load LLaMA-3-8B-Instruct

In [ ]:
from getpass import getpass
import os
os.environ['HF_TOKEN']=(os.getenv("HF_TOKEN") or getpass("Hugging Face token: ")) # Colab에서는 UI로 입력할 수 있음.
auth_token = (os.getenv("HF_TOKEN") or getpass("Hugging Face token: "))

# 모델과 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", cache_dir='./llama3/', token=auth_token)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", device_map="auto", dtype=torch.float16, cache_dir='./llama3/', token=auth_token)

### 토크나이저와 모델에서 패딩 토큰 설정

In [ ]:
tokenizer.pad_token_id = tokenizer.eos_token_id # "End of Sequence"를 나타내는 토큰의 ID이며, 시퀀스가 끝났음을 알리는 용도로 사용됨.
model.config.pad_token_id = model.config.eos_token_id

In [ ]:
tokenizer.padding_side = "left" # 패딩이 오른쪽에서 시작해서 왼쪽 방향으로 채워집니다.

##### **결과 해석**
<table>
  <thead>
    <tr>
      <th style="text-align:left; padding:4px;">태그 또는 항목</th>
      <th style="text-align:left; padding:4px;">설명</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding:4px;"><code>&lt;|begin_of_text|&gt;</code></td>
      <td style="padding:4px;">전체 텍스트나 대화의 시작을 나타냄</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|start_header_id|&gt;system&lt;|end_header_id|&gt;</code></td>
      <td style="padding:4px;">“system” 역할 메시지 표시 (AI 지침/배경 정보 포함)</td>
    </tr>
    <tr>
      <td style="padding:4px;">Cutting Knowledge Date: December 2023<br>Today Date: 26 Jul 2024</td>
      <td style="padding:4px;">대화가 발생한 시간적 맥락 제공</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|start_header_id|&gt;user&lt;|end_header_id|&gt;</code></td>
      <td style="padding:4px;">사용자가 보낸 메시지를 표시</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>Who is Vincent van Gogh?</code></td>
      <td style="padding:4px;">사용자가 제기한 질문 예시</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|start_header_id|&gt;assistant&lt;|end_header_id|&gt;</code></td>
      <td style="padding:4px;">AI 응답이 시작됨을 표시</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|eot_id|&gt;</code></td>
      <td style="padding:4px;">텍스트나 대화의 끝을 나타냄</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;userStyle&gt;Normal&lt;/userStyle&gt;</code></td>
      <td style="padding:4px;">사용자가 일반 응답을 요청함을 표시</td>
    </tr>
  </tbody>
</table>


## Llama 3을 통해 Text 생성하기

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

## Prompt 확인

In [ ]:
# 프롬프트 생성
base_prompts = generate_aspect_prompts(df)

In [ ]:
# 데이터에서 모델에 입력할 변수를 추출
instructions = [item['instruction'] for item in base_prompts]
inputs = [item['input'] for item in base_prompts]

In [15]:
instructions[0]

'Read the following review and generate the profile by explicitly including the reasoning process as part of the output.Make sure the reasoning steps are logically connected and clearly lead to the final profile.\n\n'

In [16]:
inputs[0]

'I love this<br />It is so comfortable and light weight to the point that when I put it on, it didn’t feel like I had anything on.<br />This would make a great gift<br />I love and recommend'

# LLaMA-3-8B tokenizer 기준 Input token 길이 확인

In [ ]:
# # 토큰 개수 계산 함수 정의
# def count_tokens(text):
#     return len(tokenizer.encode(text, add_special_tokens=False))

# # 컬럼 추가
# df['num_tokens'] = df['text'].progress_apply(count_tokens)

In [ ]:
# df['num_tokens'].describe()

In [ ]:
# sns.histplot(df['num_tokens'], kde=True)
# plt.title('Distribution of Token Counts')
# plt.xlabel('Token Count')
# plt.ylabel('Frequency')
# plt.show()

# Inference and Save Result

In [ ]:
generation_config = GenerationConfig(
       do_sample=True, # 모델이 텍스트를 생성할 때 확률적 샘플링을 하며, 생성 과정에서 고정된 단일 경로를 따르지 않고 더 다양한 출력을 생성 가능
        temperature=0.001, # 값이 낮을수록 높은 확률의 토큰(단어)에 더 가중치를 두어 결과가 더 결정적이 되고 값이 높을 수록 결과가 더 다양하고 무작위성이 증가
        top_p=1.0, # 출력 확률 분포에서 확률이 상위 p에 포함되는 토큰만 고려
        top_k=40, # 확률이 가장 높은 상위 k개의 토큰만 고려하여 출력
        num_beams=1, # 생성 과정에서 빔 서치(Beam Search) 알고리즘을 사용하여 여러 경로를 탐색하도록 설정
        max_new_tokens=256 # 입력 프롬프트 이후 생성될 새 토큰(단어)의 최대 개수
    )

In [18]:
# 프롬프트 생성
prompts = [
    generate_prompt(instruction=str(instruction), input=str(input))
    for instruction, input in zip(instructions, inputs)
]

# 저장 관련 설정
SAVE_INTERVAL = 1000
save_dir = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Reasoning_Text_5/"
os.makedirs(save_dir, exist_ok=True)
all_outputs = [None] * len(prompts)
chunk_index = 63

# 배치 단위 생성 및 후처리
BATCH_SIZE = 8
filled_indices = []

for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
    batch_prompt = prompts[i:i + BATCH_SIZE]

    # 토크나이징
    prompt_inputs = tokenizer(
        batch_prompt,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # 모델 응답 생성
    with torch.no_grad():
        generation_output = model.generate(
            **prompt_inputs,
            generation_config=generation_config,
            return_dict_in_generate=True,
            output_scores=False,
            return_legacy_cache=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )
    s = generation_output.sequences
    output = tokenizer.batch_decode(s, skip_special_tokens=True)
    output = [_.split('Response:\n')[-1] for _ in output]

    # 결과 저장
    for j, result in enumerate(output):
        global_idx = i + j
        all_outputs[global_idx] = result
        filled_indices.append(global_idx)

    # SAVE_INTERVAL마다 청크 저장
    if len(filled_indices) >= SAVE_INTERVAL:
        chunk_data = []
        saved_indices = []
        count = 0
        for idx in filled_indices:
            if all_outputs[idx] is not None:
                chunk_data.append(all_outputs[idx])
                saved_indices.append(idx)
                count += 1
                if count == SAVE_INTERVAL:
                    break

        save_chunk(chunk_data, chunk_index, save_dir)
        for idx in saved_indices:
            all_outputs[idx] = None
        filled_indices = [idx for idx in filled_indices if idx not in saved_indices]
        chunk_index += 1

# 남은 결과 저장
valid_outputs = [val for val in all_outputs if val is not None]
if valid_outputs:
    save_chunk(valid_outputs, chunk_index, save_dir)

 26%|██▌       | 125/482 [25:19<1:13:25, 12.34s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Reasoning_Text_5/output_chunk_063.json (1000 samples)


 52%|█████▏    | 250/482 [50:41<46:54, 12.13s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Reasoning_Text_5/output_chunk_064.json (1000 samples)


 78%|███████▊  | 375/482 [1:16:04<21:44, 12.19s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Reasoning_Text_5/output_chunk_065.json (1000 samples)


100%|██████████| 482/482 [1:37:52<00:00, 12.18s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Reasoning_Text_5/output_chunk_066.json (856 samples)


# Concat Dataset and Chunk

In [ ]:
# import json

# # 파일 경로
# input_path = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_1/output_chunk_184.json"
# output_path = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_1/output_chunk_184_cleaned.json"

# # 1. 파일 읽기
# with open(input_path, "r", encoding="utf-8") as f:
#     data = json.load(f)

# # 2. "[INVALID]" 항목 제거
# cleaned_data = [entry for entry in data if entry.strip() != "[INVALID]"]

# # 3. 다시 저장
# with open(output_path, "w", encoding="utf-8") as f:
#     json.dump(cleaned_data, f, ensure_ascii=False, indent=4)

# print(f"✅ 저장 완료: {output_path} (원래 {len(data)}개 → {len(cleaned_data)}개)")

In [31]:
# 디렉토리 경로 설정
load_dir = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Reasoning_Text_5/"

# 파일 이름 정렬
file_names = sorted([
    f for f in os.listdir(load_dir) if f.startswith("output_chunk_") and f.endswith(".json")
])

# 파일 읽기 및 DataFrame으로 로드
df_list = []
for file_name in file_names:
    file_path = os.path.join(load_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        text_df = pd.DataFrame(data)
        df_list.append(text_df)

# axis=0 방향으로 합치기 (행 방향)
combined_df = pd.concat(df_list, axis=0, ignore_index=True)

# 결과 확인 (옵션)
print(f"총 샘플 수: {len(combined_df)}")
print(combined_df.head())

총 샘플 수: 16856
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [ ]:
combined_df.columns = ['LLaMA_Output']

df = pd.concat([df, combined_df], axis=1)

In [ ]:
pd.set_option('display.max_colwidth', None)

df[['title_y', 'text', 'LLaMA_Output']].tail(20)

In [ ]:
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_reasoning_review_b5.pkl')